# VaryingVelocity — spatially varying velocity (2D + 3D)

`MagneticStandalone` · ASN 28 validation case (TRUST-LiMe).


## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs the 2D and 3D versions of this test case: ``varying_velocity``. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.


Both dimensions are run from a single shared data template in this form; the 3D mesh is converted with `Polyedriser dom`.
## What it validates

A strongly varying vortex velocity $u=(-y,x)$ is transferred onto the faces
from a functional field. With a uniform $b=b_0$, both transport terms vanish, so
$b$ must stay exactly $b_0$. This proves the varying-velocity path is correctly
instantiated and copied onto the face-centred unknown.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 7 | velocity is given data, frozen | exact invariant under a varying $u$ |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name, subdir=None):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    if subdir is None:
        subdir = "3D/coarse" if "_3d" in case.lower() else "coarse"
    path = f"{BUILD}/{subdir}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)


## Run both dimensions

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).


In [ ]:
from string import Template
from trustutils import run

NB_PROCS = 2
run.reset()

# Mesh geometries and physics/post-processing blocks are the two dimensions
# of the same src template. The 3D case uses a distinct target data name
# and build subtree, so partition zones and probe files cannot collide.
MESH_BLOCKS = {
    2: r"""Mailler dom
{
    Pave pavé
    {
        Origine              0.  0.
        Nombre_de_Noeuds     $nx $ny
        Longueurs            1.  1.
    }
    {
        Bord gauche   X = 0.    0. <= Y <= 1.
        Bord droite   X = 1.    0. <= Y <= 1.
        Bord bas      Y = 0.    0. <= X <= 1.
        Bord haut     Y = 1.    0. <= X <= 1.
    }
}""",
    3: r"""Mailler dom
{
    Pave cube
    {
        Origine              0.  0.  0.
        Nombre_de_Noeuds     $nx $ny $nz
        Longueurs            1.  1.  1.
    }
    {
        Bord gauche   X = 0.    0. <= Y <= 1.  0. <= Z <= 1.
        Bord droite   X = 1.    0. <= Y <= 1.  0. <= Z <= 1.
        Bord bas      Y = 0.    0. <= X <= 1.  0. <= Z <= 1.
        Bord haut     Y = 1.    0. <= X <= 1.  0. <= Z <= 1.
        Bord devant   Z = 0.    0. <= X <= 1.  0. <= Y <= 1.
        Bord derriere Z = 1.    0. <= X <= 1.  0. <= Y <= 1.
    }
}

Polyedriser dom""",
}
DIMENSION_BLOCKS = {
    2: {
        'boundary_block': r"""boundary_conditions
        {
            gauche  Neumann_Paroi_adiabatique
            droite  Neumann_Paroi_adiabatique
            bas     Neumann_Paroi_adiabatique
            haut    Neumann_Paroi_adiabatique
        }""",
        'post_block': r"""Postraitement
  {
    Sondes
    {
      sonde_cbx grav magnetic_field Periode 1e-6 Points 1 0.5 0.5
    }
  }""",
        'velocity_line': r"""        vitesse_magnetique     Champ_Fonc_xyz dom 2 -y x""",
    },
    3: {
        'boundary_block': r"""boundary_conditions
        {
            gauche   Neumann_Paroi_adiabatique
            droite   Neumann_Paroi_adiabatique
            bas      Neumann_Paroi_adiabatique
            haut     Neumann_Paroi_adiabatique
            devant   Neumann_Paroi_adiabatique
            derriere Neumann_Paroi_adiabatique
        }""",
        'post_block': r"""Postraitement
  {
    Sondes
    {
      sonde_cbx grav magnetic_field Periode 1e-6 Points 1 0.5 0.5 0.5
    }
  }""",
        'velocity_line': r"""        vitesse_magnetique     Champ_Fonc_xyz dom 3 -y x 0.""",
    },
}

MESHES_2D = {'coarse': {'nx': 33, 'ny': 33}, 'standard': {'nx': 65, 'ny': 65}, 'fine': {'nx': 129, 'ny': 129}}
MESHES_3D = {'coarse': {'nx': 17, 'ny': 17, 'nz': 17}, 'standard': {'nx': 33, 'ny': 33, 'nz': 33}, 'fine': {'nx': 65, 'ny': 65, 'nz': 65}}

def register_dimension(dim, meshes, prefix="", target_data=None, nb_procs=NB_PROCS, partition=True):
    registered = {}
    for level, mesh in meshes.items():
        dic = {"dim": dim,
               "mesh_block": Template(MESH_BLOCKS[dim]).substitute(**mesh),
               **DIMENSION_BLOCKS[dim]}
        target = f"{prefix}/{level}" if prefix else level
        kwargs = {"nbProcs": nb_procs, "excluNR": True}
        if target_data is not None:
            kwargs["targetData"] = target_data
        registered[level] = run.addCaseFromTemplate('varying_velocity.data', target, dic=dic, **kwargs)
        if partition:
            registered[level].partition()
    return registered

cases = register_dimension(2, MESHES_2D)
cases_3d = register_dimension(3, MESHES_3D, prefix="3D", target_data='varying_velocity_3d.data', nb_procs=NB_PROCS, partition=True)
case = cases["coarse"]
case_3d = cases_3d["coarse"]
run.printCases()
run.runCases()


## 2D checks


In [ ]:
t, b = read_son(case.dataFileName_, "SONDE_CBX")
b0 = 3.0
err = np.max(np.abs(b - b0)) / abs(b0)
print(f"max |b - b0|/b0 : {err:.3e}")
assert err < 1e-10, f"varying-velocity invariance broken (rel. error {err:.3e})"
print("OK: uniform field invariant even under a varying velocity.")


## 2D mesh convergence

The shared refinement ladder: `coarse` = the baseline mesh above (`33×33`),
`standard` = cells ×2 (`65×65`), `fine` = cells ×4 (`129×129`).

Conservation of a uniform field under a varying prescribed velocity is a
**discrete identity** — the scheme must reproduce the constant mode exactly
at any mesh. The scan asserts $\max_t |b - b_0|/|b_0| < 10^{-10}$ **at
every level**; no order is reported, the identity has no spatial error.


In [ ]:
SONDE_TAG_SCAN = "SONDE_CBX"
B0 = 3.0

def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def invariance_error(subdir, case_name):
    """max |b(t) - b0| / |b0| over all recorded times at the scan probe."""
    _, b = read_son_dir(subdir, case_name, SONDE_TAG_SCAN)
    return float(np.max(np.abs(b - B0)) / abs(B0))

scan = {"coarse": ("coarse", case.dataFileName_),
        "standard": (f"standard", cases["standard"].dataFileName_),
        "fine": (f"fine", cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (33, 33), "standard": (65, 65), "fine": (129, 129)}
scan_err = {lvl: invariance_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes      max |b - b0|/b0")
for lvl in ("coarse", "standard", "fine"):
    nx, ny = scan_nodes[lvl]
    print(f"{lvl:<9} {nx:>4}x{ny:<4} {scan_err[lvl]:.3e}")
    assert scan_err[lvl] < 1e-10, \
        f"varying-velocity invariance broken at {lvl} (rel. error {scan_err[lvl]:.3e})"
print("OK: uniform-field invariance under a varying velocity holds at every level.")

## 3D checks


In [ ]:
t, b = read_son(case_3d.dataFileName_, "SONDE_CBX")
b0 = 3.0
err = np.max(np.abs(b - b0)) / abs(b0)
print(f"max |b - b0|/b0 : {err:.3e}")
assert err < 1e-10, f"varying-velocity invariance broken (rel. error {err:.3e})"
print("OK: uniform field invariant even under a varying velocity.")


## 3D mesh convergence

The shared refinement ladder: `coarse` = the baseline mesh above (`17×17×17`),
`standard` = cells ×2 (`33×33×33`), `fine` = cells ×4 (`65×65×65`).

Conservation of a uniform field under a varying prescribed velocity is a **discrete identity**: the scan asserts max\|b − b₀\|/\|b₀\| < 10⁻¹⁰ **at every level** — no order is reported.

In [ ]:
SONDE_TAG_SCAN = "SONDE_CBX"
B0 = 3.0

def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def invariance_error(subdir, case_name):
    _, b = read_son_dir(subdir, case_name, SONDE_TAG_SCAN)
    return float(np.max(np.abs(b - B0)) / abs(B0))

scan = {"coarse": ("3D/coarse", case_3d.dataFileName_),
        "standard": ("3D/standard", cases_3d["standard"].dataFileName_),
        "fine": ("3D/fine", cases_3d["fine"].dataFileName_)}
scan_nodes = {"coarse": (17, 17, 17), "standard": (33, 33, 33), "fine": (65, 65, 65)}
scan_err = {lvl: invariance_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes        max |b - b0|/b0")
for lvl in ("coarse", "standard", "fine"):
    nx, ny, nz = scan_nodes[lvl]
    print(f"{lvl:<9} {nx:>3}x{ny}x{nz:<3} {scan_err[lvl]:.3e}")
    assert scan_err[lvl] < 1e-10, f"varying-velocity invariance broken at {lvl}: {scan_err[lvl]:.3e}"
print("OK: uniform-field invariance under a varying velocity holds at every level.")

## Computer Performance


In [ ]:
run.tablePerf()